In [236]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [237]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')
df['주차'].info()

<class 'pandas.core.series.Series'>
Index: 51279 entries, 0 to 51278
Series name: 주차
Non-Null Count  Dtype         
--------------  -----         
51279 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 801.2 KB


In [238]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [239]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [240]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

sex_df = df[df['성별'].str.contains('기타')]
sex_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

- Q. 시즌 이월 확인 코드?

In [241]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 ( -12578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()

시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

In [242]:
print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

df.head(3)

[행/컬럼 갯수]
행: 51267, 컬럼: 40



,기간,주차,라인,성별,기획년도,시즌이월,상품년차,시즌,복종,소품종,CAT,총입고수량,총입고원가,총입고택가,총출고수량,총출고원가,총출고택가,판매액,판매수량,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,물류재고수량,물류재고원가,물류재고택가,매장재고수량,매장재고원가,매장재고택가,재고수량,재고원가,재고택가,기간입고수량,기간입고원가,기간입고택가,기간출고수량,기간출고원가,기간출고택가
0,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,우븐 셔츠,캐쥬얼셔츠,02_SHIRTS,2247,19893640,157065300,1009,8933103,70529100,-124850,0,0,0,84850,3,26560,209700,1238,10960537,86536200,1006,8906543,70319400,2244,19867080,156855600,0,0,0,53,469228,3704700
1,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,사계절,소품,양말,10_ACC/ETC,14000,11480000,46200000,10775,8835500,35557500,2033156,617,505940,2036100,10270838,3127,2564140,10319100,3225,2644500,10642500,7648,6271360,25238400,10873,8915860,35880900,0,0,0,454,372280,1498200
2,당해,2021-01-03,ZB,1:남성,2021,01_시즌,당해년도,봄,니트 셔츠,라운드,01_KNIT,20076,178239296,1403312400,5940,52736683,415206000,18981424,401,3559559,28029900,18981424,401,3560170,28029900,14136,125502613,988106400,5539,49176513,387176100,19675,174679125,1375282500,0,0,0,5940,52713656,415206000


In [243]:
df['성별'].unique()

array(['1:남성', '4:기타', '3:남녀공용', '2:여성'], dtype=object)

In [244]:
# df[df['성별'] == '2:여성']

# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

재언 : 사용 컬럼에 성별이 없어서 추가했어요

In [245]:
# 총입고수량/입고원가/입고택가  -> 전처리 (-162행)
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 

#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['주차','라인', '성별','기획년도','시즌이월','시즌','복종','소품종', '카테고리', '총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가']]

print((num_df[f'총입고수량'] == 0).sum())
filtered_df = num_df[(df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

162
[행/컬럼 갯수]
행: 51105, 컬럼: 23



In [246]:
# 재고수량/재고원가/재고택가 정합성 확인 -> 재고 관련 컬럼 삭제 
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")

[행/컬럼 갯수]
행: 51105, 컬럼: 20



# 여기서부터 재언

### 확인

In [247]:
filtered_df

,주차,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가
0,2021-01-03,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700
1,2021-01-03,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,14000,11480000,46200000,617,2033156,505940,2036100,10270838,3127,2564140,10319100
2,2021-01-03,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900
3,2021-01-03,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,7652,49967560,381834800,174,8450445,1136220,8682600,85204225,1733,11316490,86476700
4,2021-01-03,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,405,26409229,153495000,6,1682760,391248,2274000,16797003,60,3912478,22740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,2024-12-29,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,1504,11731200,73696000,4,196000,31200,196000,38217400,789,6154200,38661000
51275,2024-12-29,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,1404,38437753,209196000,25,2227550,684433,3725000,28603530,272,7446630,40528000
51276,2024-12-29,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,1926,28828072,190674000,0,0,0,0,56664950,1241,18575097,122859000
51277,2024-12-29,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,19614,139893880,1567158600,0,0,0,0,302854418,19428,138567263,1552297200


In [248]:
filtered_df['라인'].unique()

array(['ZB', 'ZA', 'ZE', 'ZD', 'ZC', 'ZF', 'ZG'], dtype=object)

In [249]:
filtered_df['성별'].unique()

array(['1:남성', '4:기타', '3:남녀공용', '2:여성'], dtype=object)

In [250]:
filtered_df[filtered_df['성별'] == '2:여성']['라인'].unique()

array(['ZB', 'ZD', 'ZF'], dtype=object)

In [251]:
female_df = filtered_df[filtered_df['성별'] == '2:여성']
female_df[female_df['라인']=='ZB']

,주차,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가
18642,2022-09-11,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
18926,2022-09-18,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
19382,2022-09-25,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
19718,2022-10-02,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
20070,2022-10-09,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
20467,2022-10-16,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
20746,2022-10-23,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
21146,2022-10-30,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
21466,2022-11-06,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0
21998,2022-11-13,ZB,2:여성,2022,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,5500,6600000,32450000,0,0,0,0,0,0,0,0


## 성별 == 여성 제외

In [252]:
# using_df = filtered_df[~(filtered_df['성별'] == '2:여성')]
# using_df

### 확인

In [253]:
# using_df['성별'].unique()

## 사용할 컬럼만 다시 추출

In [254]:
filtered_df.head()

,주차,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가
0,2021-01-03,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2247,19893640,157065300,0,-124850,0,0,84850,3,26560,209700
1,2021-01-03,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,14000,11480000,46200000,617,2033156,505940,2036100,10270838,3127,2564140,10319100
2,2021-01-03,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,20076,178239296,1403312400,401,18981424,3559559,28029900,18981424,401,3560170,28029900
3,2021-01-03,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,7652,49967560,381834800,174,8450445,1136220,8682600,85204225,1733,11316490,86476700
4,2021-01-03,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,405,26409229,153495000,6,1682760,391248,2274000,16797003,60,3912478,22740000


In [255]:
using_df = filtered_df.copy()
using_df = using_df[['라인', '성별', '기획년도', '시즌이월', '시즌', '복종', '소품종', '카테고리', '주차', '총입고수량', '총입고원가', '총입고택가', '판매수량', '판매액']]
using_df

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2024-12-29,1504,11731200,73696000,4,196000
51275,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,2024-12-29,1404,38437753,209196000,25,2227550
51276,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,2024-12-29,1926,28828072,190674000,0,0
51277,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,2024-12-29,19614,139893880,1567158600,0,0


## 제품 개당 원가/택가/실판매가 컬럼 생성

In [256]:
using_df['개당원가'] = (using_df['총입고원가'] / using_df['총입고수량']).round(0)
using_df['개당택가'] = (using_df['총입고택가'] / using_df['총입고수량']).round(0)
using_df['개당실판매가'] = (using_df['판매액'] / using_df['판매수량']).round(0)
using_df

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,개당원가,개당택가,개당실판매가
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,8853.0,69900.0,-inf
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51274,ZA,1:남성,2024,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2024-12-29,1504,11731200,73696000,4,196000,7800.0,49000.0,49000.0
51275,ZB,1:남성,2024,01_시즌,겨울,수트,수트팬츠,겨울_수트_수트팬츠_ZB,2024-12-29,1404,38437753,209196000,25,2227550,27377.0,149000.0,89102.0
51276,ZB,1:남성,2024,02_이월,여름,스웨터,라운드,여름_스웨터_라운드_ZB,2024-12-29,1926,28828072,190674000,0,0,14968.0,99000.0,NaN
51277,ZE,1:남성,2024,01_시즌,사계절,우븐 셔츠,드레스셔츠,사계절_우븐 셔츠_드레스셔츠_ZE,2024-12-29,19614,139893880,1567158600,0,0,7132.0,79900.0,NaN


### inf, nan 값 확인 및 inf를 nan으로 대체

In [257]:
print(using_df['개당원가'].apply(np.isinf).sum())
print(using_df['개당택가'].apply(np.isinf).sum())
print(using_df['개당실판매가'].apply(np.isinf).sum())

0
0
264


In [258]:
print(using_df['개당원가'].isna().sum())
print(using_df['개당택가'].isna().sum())
print(using_df['개당실판매가'].isna().sum())

0
0
7554


In [259]:
using_df['개당실판매가'].replace([np.inf, -np.inf], np.nan, inplace=True)
using_df['개당실판매가'].apply(np.isinf).sum()

0

In [260]:
print(using_df['개당실판매가'].isna().sum())

7818


### 실판매가 == 0 or  <0 확인

In [261]:
using_df.head()

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,개당원가,개당택가,개당실판매가
0,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-03,2247,19893640,157065300,0,-124850,8853.0,69900.0,NaN
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0


In [262]:
using_df[using_df['개당실판매가'] < 0]

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,개당원가,개당택가,개당실판매가
96,ZB,1:남성,2021,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZB,2021-01-10,2995,21348695,209350500,16,-351690,7128.0,69900.0,-21981.0
202,ZB,1:남성,2021,01_시즌,봄,스웨터,오픈형(CARDIGAN),봄_스웨터_오픈형(CARDIGAN)_ZB,2021-01-17,3259,49575364,521114100,44,-5050,15212.0,159900.0,-115.0
231,ZA,1:남성,2021,01_시즌,봄,수트,수트팬츠,봄_수트_수트팬츠_ZA,2021-01-17,489,18401014,107091000,1,-21900,37630.0,219000.0,-21900.0
332,ZB,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZB,2021-01-31,1516,61021238,301684000,1,-278600,40251.0,199000.0,-278600.0
335,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-31,27000,169399202,1887300000,73,-7087970,6274.0,69900.0,-97095.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49482,ZB,1:남성,2024,02_이월,여름,스웨터,T-에리,여름_스웨터_T-에리_ZB,2024-12-01,22000,210669018,1958000000,-2,55100,9576.0,89000.0,-27550.0
49500,ZE,1:남성,2024,02_이월,여름,니트 셔츠,라운드,여름_니트 셔츠_라운드_ZE,2024-12-01,16445,97123752,1299155000,-2,14310,5906.0,79000.0,-7155.0
49552,ZE,1:남성,2024,02_이월,여름,니트 셔츠,라운드,여름_니트 셔츠_라운드_ZE,2024-12-01,40424,192786773,3193496000,-1,64997,4769.0,79000.0,-64997.0
49680,ZE,1:남성,2024,02_이월,여름,니트 셔츠,티에리,여름_니트 셔츠_티에리_ZE,2024-12-01,29028,232479138,2293212000,-3,121756,8009.0,79000.0,-40585.0


In [263]:
using_df[using_df['개당실판매가'] == 0]

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,개당원가,개당택가,개당실판매가
4398,ZA,1:남성,2021,02_이월,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-07-11,1509,78118694,481371000,-1,0,51769.0,319000.0,-0.0
4556,ZA,1:남성,2021,02_이월,봄,수트,수트팬츠,봄_수트_수트팬츠_ZA,2021-07-11,489,18401014,107091000,-1,0,37630.0,219000.0,-0.0
5582,ZD,1:남성,2021,02_이월,봄,우븐 셔츠,캐쥬얼셔츠,봄_우븐 셔츠_캐쥬얼셔츠_ZD,2021-08-15,284,2552194,19851600,1,0,8987.0,69900.0,0.0
7034,ZB,1:남성,2021,01_시즌,여름,팬츠,반바지,여름_팬츠_반바지_ZB,2021-09-19,15204,128114423,910719600,-2,0,8426.0,59900.0,-0.0
7566,ZE,1:남성,2021,02_이월,여름,자켓,싱글재킷,여름_자켓_싱글재킷_ZE,2021-10-03,2697,57943036,536703000,-26,0,21484.0,199000.0,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49549,ZE,1:남성,2024,02_이월,여름,자켓,싱글재킷,여름_자켓_싱글재킷_ZE,2024-12-01,6276,169736766,1876524000,-1,0,27045.0,299000.0,-0.0
49553,ZB,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZB,2024-12-01,16579,201618799,1641321000,-4,0,12161.0,99000.0,-0.0
49677,ZE,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZE,2024-12-01,2554,28532326,252846000,-1,0,11172.0,99000.0,-0.0
49977,ZB,1:남성,2024,02_이월,여름,팬츠,팬츠(일반),여름_팬츠_팬츠(일반)_ZB,2024-12-08,3947,43506618,390753000,-1,0,11023.0,99000.0,-0.0


## [기획년도,주차,카테고리] 집계 시작 1단계

# (채영) 성별 컬럼 누락되어 추가

In [264]:
agg_dict = {
    '라인': 'first', 
    '시즌이월': 'first', 
    '시즌': 'first', 
    '복종': 'first', 
    '소품종': 'first',
    '성별': 'first',  
    '총입고수량': 'sum',
    '판매수량' : 'sum', 
    '판매액': 'sum', 
    '개당택가': 'mean', 
    '개당원가': 'mean'
}

grouped_df = using_df.groupby(['기획년도', '주차', '카테고리']).agg(agg_dict).reset_index()
grouped_df

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,개당택가,개당원가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000.0,204500.000000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900.0,8878.000000
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900.0,6179.000000
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000.0,56150.333333
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000.0,40583.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21340,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,ZB,02_이월,여름,팬츠,팬츠(일반),1:남성,84786,0,0,99000.0,11930.500000
21341,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,ZD,02_이월,여름,팬츠,팬츠(일반),1:남성,2394,0,0,99000.0,13104.000000
21342,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,ZE,02_이월,여름,팬츠,팬츠(일반),1:남성,38679,0,0,105000.0,10790.400000
21343,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZF,ZF,02_이월,여름,팬츠,팬츠(일반),2:여성,1237,0,0,99000.0,16868.000000


In [265]:
grouped_df['총입고원가'] = grouped_df['총입고수량'] * grouped_df['개당원가']
grouped_df['총입고택가'] = grouped_df['총입고수량'] * grouped_df['개당택가']
grouped_df['매출원가계'] = grouped_df['판매수량'] * grouped_df['개당원가']
grouped_df['판매택가계'] = grouped_df['판매수량'] * grouped_df['개당택가']
grouped_df

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,개당택가,개당원가,총입고원가,총입고택가,매출원가계,판매택가계
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000.0,204500.000000,3.272000e+08,1.598400e+09,1.083850e+07,52947000.0
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900.0,8878.000000,1.782347e+08,1.403312e+09,3.560078e+06,28029900.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900.0,6179.000000,1.059575e+08,1.198645e+09,4.325300e+04,489300.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000.0,56150.333333,2.458262e+08,1.484142e+09,6.120386e+06,36951000.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000.0,40583.200000,3.397626e+08,1.666028e+09,7.873141e+06,38606000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21340,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,ZB,02_이월,여름,팬츠,팬츠(일반),1:남성,84786,0,0,99000.0,11930.500000,1.011539e+09,8.393814e+09,0.000000e+00,0.0
21341,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,ZD,02_이월,여름,팬츠,팬츠(일반),1:남성,2394,0,0,99000.0,13104.000000,3.137098e+07,2.370060e+08,0.000000e+00,0.0
21342,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,ZE,02_이월,여름,팬츠,팬츠(일반),1:남성,38679,0,0,105000.0,10790.400000,4.173619e+08,4.061295e+09,0.000000e+00,0.0
21343,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZF,ZF,02_이월,여름,팬츠,팬츠(일반),2:여성,1237,0,0,99000.0,16868.000000,2.086572e+07,1.224630e+08,0.000000e+00,0.0


# (채영) 총입고원가/택가 음수값 확인 및 int32 -> int 64로 변환 

In [267]:
columns_to_round = ['개당택가','개당원가', '총입고원가', '총입고택가', '매출원가계', '판매택가계']

grouped_df[columns_to_round] = grouped_df[columns_to_round].round(0).astype('int64')
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,개당택가,개당원가,총입고원가,총입고택가,매출원가계,판매택가계
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000


### 중간 확인

In [269]:
grouped_df.dtypes

기획년도              int64
주차       datetime64[ns]
카테고리             object
라인               object
시즌이월             object
시즌               object
복종               object
소품종              object
성별               object
총입고수량             int64
판매수량              int64
판매액               int64
개당택가              int64
개당원가              int64
총입고원가             int64
총입고택가             int64
매출원가계             int64
판매택가계             int64
dtype: object

In [270]:
grouped_df[grouped_df['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA']

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,개당택가,개당원가,총입고원가,총입고택가,매출원가계,판매택가계
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000
31,2021,2021-01-10,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,108,44542500,999000,204500,327200000,1598400000,22086000,107892000
69,2021,2021-01-17,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,79,23101100,999000,204500,327200000,1598400000,16155500,78921000
109,2021,2021-01-24,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,51,24770500,999000,204500,327200000,1598400000,10429500,50949000
149,2021,2021-01-31,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,170,83960000,999000,204500,327200000,1598400000,34765000,169830000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15211,2023,2023-12-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,5,2495000,1149000,265036,93292672,404448000,1325180,5745000
15374,2023,2023-12-10,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,0,0,1149000,265036,93292672,404448000,0,0
15537,2023,2023-12-17,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,-1,-399000,1149000,265036,93292672,404448000,-265036,-1149000
15700,2023,2023-12-24,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,6,3594000,1149000,265036,93292672,404448000,1590216,6894000


## [기획년도,주차,카테고리] 집계 시작 2단계

In [271]:
using_df_실판가_over_zero = using_df[using_df['개당실판매가'] > 0]
주차별_실판매가_df = using_df_실판가_over_zero.groupby(['기획년도', '주차', '카테고리'])['개당실판매가'].mean().reset_index()
주차별_실판매가_df.rename(columns={'개당실판매가': '주차별_평균_실판매가'}, inplace=True)
주차별_실판매가_df.head()

,기획년도,주차,카테고리,주차별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,987705.5
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,47335.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,30645.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,207972.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,199527.8


In [272]:
grouped_df = grouped_df.merge(주차별_실판매가_df, on=["기획년도", "주차", "카테고리"], how="left")
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,개당택가,개당원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8


In [273]:
grouped_df.rename(columns={
    "개당택가": "평균택가",
    "개당원가": "평균원가",
}, inplace=True)
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8


### 중간확인

In [274]:
grouped_df[grouped_df['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA']

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5
31,2021,2021-01-10,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,108,44542500,999000,204500,327200000,1598400000,22086000,107892000,392363.0
69,2021,2021-01-17,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,79,23101100,999000,204500,327200000,1598400000,16155500,78921000,294492.0
109,2021,2021-01-24,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,51,24770500,999000,204500,327200000,1598400000,10429500,50949000,485429.0
149,2021,2021-01-31,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,170,83960000,999000,204500,327200000,1598400000,34765000,169830000,493421.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15211,2023,2023-12-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,5,2495000,1149000,265036,93292672,404448000,1325180,5745000,499000.0
15374,2023,2023-12-10,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,0,0,1149000,265036,93292672,404448000,0,0,NaN
15537,2023,2023-12-17,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,-1,-399000,1149000,265036,93292672,404448000,-265036,-1149000,399000.0
15700,2023,2023-12-24,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,6,3594000,1149000,265036,93292672,404448000,1590216,6894000,599000.0


In [275]:
grouped_df[(grouped_df['카테고리'] == '봄_가죽&FUR_가죽점퍼_ZA') & (grouped_df['판매수량'] >= 0) & (grouped_df['판매액'] < 0)]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가
2220,2021,2021-08-22,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,1600,14,-7346970,999000,204500,327200000,1598400000,2863000,13986000,657433.5
3203,2021,2021-10-17,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,1600,17,-1451480,999000,204500,327200000,1598400000,3476500,16983000,78752.0
7249,2022,2022-08-14,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,1298,0,-300400,999000,183300,237923400,1296702000,0,0,NaN
13122,2023,2023-09-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,02_이월,봄,가죽&FUR,가죽점퍼,1:남성,352,1,-1000,1149000,265036,93292672,404448000,265036,1149000,199000.0


## [기획년도,주차,카테고리] 집계 시작 3단계

In [276]:
using_df_실판가_over_zero['월'] = using_df_실판가_over_zero['주차'].dt.month
using_df_실판가_over_zero.head()

,라인,성별,기획년도,시즌이월,시즌,복종,소품종,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,개당원가,개당택가,개당실판매가,월
1,ZB,1:남성,2021,01_시즌,사계절,소품,양말,사계절_소품_양말_ZB,2021-01-03,14000,11480000,46200000,617,2033156,820.0,3300.0,3295.0,1
2,ZB,1:남성,2021,01_시즌,봄,니트 셔츠,라운드,봄_니트 셔츠_라운드_ZB,2021-01-03,20076,178239296,1403312400,401,18981424,8878.0,69900.0,47335.0,1
3,ZA,1:남성,2021,01_시즌,사계절,소품,타이,사계절_소품_타이_ZA,2021-01-03,7652,49967560,381834800,174,8450445,6530.0,49900.0,48566.0,1
4,ZA,1:남성,2021,01_시즌,봄,수트,블레이져(수트),봄_수트_블레이져(수트)_ZA,2021-01-03,405,26409229,153495000,6,1682760,65208.0,379000.0,280460.0,1
5,ZB,1:남성,2021,01_시즌,봄,수트,수트팬츠,봄_수트_수트팬츠_ZB,2021-01-03,1754,33478840,173646000,24,1682800,19087.0,99000.0,70117.0,1


In [277]:
월별_실판매가_df = using_df_실판가_over_zero.groupby(['기획년도', '월', '카테고리'])['개당실판매가'].mean().reset_index()
월별_실판매가_df.rename(columns={'개당실판매가': '월별_평균_실판매가'}, inplace=True)
월별_실판매가_df.head()


,기획년도,월,카테고리,월별_평균_실판매가
0,2021,1,봄_가죽&FUR_가죽점퍼_ZA,530682.100000
1,2021,1,봄_니트 셔츠_라운드_ZB,41773.500000
2,2021,1,봄_니트 셔츠_라운드_ZE,24672.600000
3,2021,1,봄_수트_블레이져(수트)_ZA,188352.533333
4,2021,1,봄_수트_블레이져(수트)_ZB,160160.125000


In [278]:
grouped_df['월'] = grouped_df['주차'].dt.month
grouped_df = grouped_df.merge(월별_실판매가_df, on=['기획년도', '월', '카테고리'], how="left")
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5,1,530682.100000
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41773.500000
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24672.600000
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188352.533333
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8,1,160160.125000


### 중간 확인

In [279]:
grouped_df['월별_평균_실판매가'].isna().sum()

2155

In [280]:
grouped_df['주차별_평균_실판매가'].isna().sum()

3806

## [기획년도,주차,카테고리] 집계 시작 4단계

In [281]:
시즌별_실판매가_df = using_df_실판가_over_zero.groupby(['기획년도', '시즌이월', '카테고리'])['개당실판매가'].mean().reset_index()
시즌별_실판매가_df.rename(columns={'개당실판매가': '시즌별_평균_실판매가'}, inplace=True)
시즌별_실판매가_df.head()


,기획년도,시즌이월,카테고리,시즌별_평균_실판매가
0,2021,01_시즌,가을_니트 셔츠_라운드_ZB,31895.351351
1,2021,01_시즌,가을_니트 셔츠_라운드_ZD,26250.000000
2,2021,01_시즌,가을_수트_블레이져(수트)_ZB,183882.369231
3,2021,01_시즌,가을_수트_블레이져(수트)_ZC,177987.047619
4,2021,01_시즌,가을_수트_수트팬츠_ZB,97927.876923


In [282]:
시즌별_실판매가_df['시즌별_평균_실판매가'].isna().sum()

0

In [283]:
grouped_df = grouped_df.merge(시즌별_실판매가_df, on=['기획년도', '시즌이월', '카테고리'], how="left")
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5,1,530682.100000,455693.139535
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41773.500000,27610.220339
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24672.600000,20711.514286
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188352.533333,150188.410959
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8,1,160160.125000,128110.431193


### 중간 확인

In [284]:
grouped_df['시즌별_평균_실판매가'].isna().sum()

575

In [285]:
grouped_df[grouped_df['시즌별_평균_실판매가'].isna()]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가
1318,2021,2021-06-06,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN
1326,2021,2021-06-06,봄_점퍼_패딩점퍼_ZD,ZD,02_이월,봄,점퍼,패딩점퍼,1:남성,300,0,0,199000,27004,8101200,59700000,0,0,NaN,6,NaN,NaN
1390,2021,2021-06-13,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN
1398,2021,2021-06-13,봄_점퍼_패딩점퍼_ZD,ZD,02_이월,봄,점퍼,패딩점퍼,1:남성,300,0,0,199000,27004,8101200,59700000,0,0,NaN,6,NaN,NaN
1462,2021,2021-06-20,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21300,2024,2024-12-29,여름_니트 셔츠_티에리_ZD,ZD,02_이월,여름,니트 셔츠,티에리,1:남성,3000,0,0,79000,10263,30789000,237000000,0,0,NaN,12,NaN,NaN
21325,2024,2024-12-29,여름_자켓_싱글재킷_ZD,ZD,02_이월,여름,자켓,싱글재킷,1:남성,1223,0,0,299000,28768,35183264,365677000,0,0,NaN,12,NaN,NaN
21330,2024,2024-12-29,여름_점퍼_점퍼_ZE,ZE,02_이월,여름,점퍼,점퍼,1:남성,1983,0,0,249000,24247,48081801,493767000,0,0,NaN,12,NaN,NaN
21335,2024,2024-12-29,여름_팬츠_반바지_ZD,ZD,02_이월,여름,팬츠,반바지,1:남성,5029,0,0,79000,6079,30571291,397291000,0,0,NaN,12,NaN,NaN


## 실판매가 컬럼 추가

In [286]:
grouped_df['실판매가'] = grouped_df['판매액'] / grouped_df['판매수량']
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5,1,530682.100000,455693.139535,986743.396226
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41773.500000,27610.220339,47335.221945
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24672.600000,20711.514286,30645.142857
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188352.533333,150188.410959,180600.000000
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8,1,160160.125000,128110.431193,154077.015464


### 중간 확인

In [287]:
grouped_df[grouped_df['실판매가'] <= 0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가
343,2021,2021-02-21,여름_우븐 셔츠_드레스셔츠_ZB,ZB,01_시즌,여름,우븐 셔츠,드레스셔츠,1:남성,42909,95,-110210,69900,6329,271571061,2999339100,601255,6640500,NaN,2,50254.000000,24611.448718,-1.160105e+03
407,2021,2021-02-28,여름_점퍼_점퍼_ZE,ZE,01_시즌,여름,점퍼,점퍼,1:남성,4022,1,-8875,249000,19370,77906140,1001478000,19370,249000,NaN,2,72441.000000,47718.446429,-8.875000e+03
974,2021,2021-05-02,봄_조끼_패딩베스트_ZB,ZB,01_시즌,봄,조끼,패딩베스트,1:남성,10202,26,-352207,79000,10099,103029998,805958000,262574,2054000,NaN,5,11972.250000,17099.000000,-1.354642e+04
1045,2021,2021-05-09,봄_코트_더블코트_ZB,ZB,01_시즌,봄,코트,더블코트,1:남성,3377,9,-110449,329000,34815,117570255,1111033000,313335,2961000,NaN,5,93452.750000,144883.047619,-1.227211e+04
1088,2021,2021-05-09,여름_팬츠_반바지_ZB,ZB,01_시즌,여름,팬츠,반바지,1:남성,15800,7,-185152,54900,6646,105014700,867420000,46526,384300,NaN,5,36940.000000,26361.360000,-2.645029e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20712,2024,2024-12-01,여름_팬츠_반바지_ZE,ZE,02_이월,여름,팬츠,반바지,1:남성,5031,-2,0,79000,7848,39483288,397449000,-15696,-158000,NaN,12,NaN,14411.000000,-0.000000e+00
20716,2024,2024-12-01,여름_팬츠_팬츠(일반)_ZB,ZB,02_이월,여름,팬츠,팬츠(일반),1:남성,84786,-2,41838,99000,11930,1011539373,8393814000,-23861,-198000,20919.0,12,35369.150000,36938.779412,-2.091900e+04
21156,2024,2024-12-22,여름_스웨터_T-에리_ZF,ZF,02_이월,여름,스웨터,T-에리,2:여성,1125,0,-6000,94000,15254,17160750,105750000,0,0,56400.0,12,55400.000000,55900.000000,-inf
21239,2024,2024-12-29,겨울_자켓_싱글재킷_ZA,ZA,01_시즌,겨울,자켓,싱글재킷,1:남성,10296,16,-4177761,332333,42772,440377080,3421704000,684347,5317333,164285.0,12,165827.866667,214754.604167,-2.611101e+05


In [288]:
grouped_df[grouped_df['주차별_평균_실판매가'] <= 0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가


In [289]:
print(grouped_df['실판매가'].apply(np.isinf).sum())

97


## 실판매가 이상치 대체

In [290]:
grouped_df['실판매가'].replace([np.inf, -np.inf], np.nan, inplace=True)
grouped_df['실판매가'].apply(np.isinf).sum()

0

In [291]:
grouped_df['실판매가'] = grouped_df['실판매가'].mask((grouped_df['실판매가'] <= 0) | (grouped_df['실판매가'].isna()), grouped_df['주차별_평균_실판매가'])

grouped_df['실판매가'] = grouped_df['실판매가'].fillna(grouped_df['월별_평균_실판매가'])

grouped_df['실판매가'] = grouped_df['실판매가'].fillna(grouped_df['시즌별_평균_실판매가'])

grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5,1,530682.100000,455693.139535,986743.396226
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41773.500000,27610.220339,47335.221945
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24672.600000,20711.514286,30645.142857
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188352.533333,150188.410959,180600.000000
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8,1,160160.125000,128110.431193,154077.015464


### 중간 확인

In [292]:
grouped_df[grouped_df['실판매가'] <= 0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가


실판매가를 평균값으로 다 대체했는데도 NaN인 경우를 뽑아봤는데, 그냥 시즌 마감하고 쭉~ 판매가 없는 내역인거 같음. 이상없겠쥬?

In [293]:
grouped_df[grouped_df['실판매가'].isna()]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가
1318,2021,2021-06-06,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN,NaN
1326,2021,2021-06-06,봄_점퍼_패딩점퍼_ZD,ZD,02_이월,봄,점퍼,패딩점퍼,1:남성,300,0,0,199000,27004,8101200,59700000,0,0,NaN,6,NaN,NaN,NaN
1390,2021,2021-06-13,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN,NaN
1398,2021,2021-06-13,봄_점퍼_패딩점퍼_ZD,ZD,02_이월,봄,점퍼,패딩점퍼,1:남성,300,0,0,199000,27004,8101200,59700000,0,0,NaN,6,NaN,NaN,NaN
1462,2021,2021-06-20,봄_스웨터_라운드_ZD,ZD,02_이월,봄,스웨터,라운드,1:남성,1603,0,0,89900,9315,14931945,144109700,0,0,NaN,6,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21300,2024,2024-12-29,여름_니트 셔츠_티에리_ZD,ZD,02_이월,여름,니트 셔츠,티에리,1:남성,3000,0,0,79000,10263,30789000,237000000,0,0,NaN,12,NaN,NaN,NaN
21325,2024,2024-12-29,여름_자켓_싱글재킷_ZD,ZD,02_이월,여름,자켓,싱글재킷,1:남성,1223,0,0,299000,28768,35183264,365677000,0,0,NaN,12,NaN,NaN,NaN
21330,2024,2024-12-29,여름_점퍼_점퍼_ZE,ZE,02_이월,여름,점퍼,점퍼,1:남성,1983,0,0,249000,24247,48081801,493767000,0,0,NaN,12,NaN,NaN,NaN
21335,2024,2024-12-29,여름_팬츠_반바지_ZD,ZD,02_이월,여름,팬츠,반바지,1:남성,5029,0,0,79000,6079,30571291,397291000,0,0,NaN,12,NaN,NaN,NaN


In [294]:
grouped_df['시즌이월'].unique()

array(['01_시즌', '02_이월'], dtype=object)

In [295]:
grouped_df[(grouped_df['시즌이월']=='01_시즌') & (grouped_df['실판매가'].isna())]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가
4666,2022,2022-01-02,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,1,NaN,NaN,NaN
4704,2022,2022-01-09,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,1,NaN,NaN,NaN
4746,2022,2022-01-16,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,1,NaN,NaN,NaN
4792,2022,2022-01-23,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,1,NaN,NaN,NaN
4842,2022,2022-01-30,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,1,NaN,NaN,NaN
4898,2022,2022-02-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,2,NaN,NaN,NaN
4956,2022,2022-02-13,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,2,NaN,NaN,NaN
5022,2022,2022-02-20,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,2,NaN,NaN,NaN
5089,2022,2022-02-27,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,2,NaN,NaN,NaN
5160,2022,2022-03-06,봄_우븐 셔츠_캐쥬얼셔츠_ZD,ZD,01_시즌,봄,우븐 셔츠,캐쥬얼셔츠,1:남성,300,0,0,129900,5221,1566300,38970000,0,0,NaN,3,NaN,NaN,NaN


## 할인율 컬럼 추가

In [296]:
grouped_df['할인율'] = ((grouped_df['평균택가'] - grouped_df['실판매가']) / grouped_df['평균택가']) * 100
grouped_df

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987705.5,1,530682.100000,455693.139535,986743.396226,1.226887
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41773.500000,27610.220339,47335.221945,32.281514
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24672.600000,20711.514286,30645.142857,56.158594
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188352.533333,150188.410959,180600.000000,46.725664
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199527.8,1,160160.125000,128110.431193,154077.015464,22.574364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21340,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZB,ZB,02_이월,여름,팬츠,팬츠(일반),1:남성,84786,0,0,99000,11930,1011539373,8393814000,0,0,NaN,12,35369.150000,36938.779412,35369.150000,64.273586
21341,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZD,ZD,02_이월,여름,팬츠,팬츠(일반),1:남성,2394,0,0,99000,13104,31370976,237006000,0,0,NaN,12,NaN,NaN,NaN,NaN
21342,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZE,ZE,02_이월,여름,팬츠,팬츠(일반),1:남성,38679,0,0,105000,10790,417361882,4061295000,0,0,NaN,12,45195.000000,29690.444444,45195.000000,56.957143
21343,2024,2024-12-29,여름_팬츠_팬츠(일반)_ZF,ZF,02_이월,여름,팬츠,팬츠(일반),2:여성,1237,0,0,99000,16868,20865716,122463000,0,0,NaN,12,61875.000000,57962.666667,61875.000000,37.500000


In [297]:
print(grouped_df['주차별_평균_실판매가'].apply(np.isinf).sum())
print(grouped_df['월별_평균_실판매가'].apply(np.isinf).sum())
print(grouped_df['시즌별_평균_실판매가'].apply(np.isinf).sum())
print(grouped_df['실판매가'].apply(np.isinf).sum())
print(grouped_df['할인율'].apply(np.isinf).sum())

0
0
0
0
0


In [298]:
columns_to_round = ['주차별_평균_실판매가', '월별_평균_실판매가', '시즌별_평균_실판매가', '실판매가', '할인율']

grouped_df[columns_to_round] = grouped_df[columns_to_round].round(0)
#.astype('Int64')

In [299]:
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693.0,986743.0,1.0
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41774.0,27610.0,47335.0,32.0
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24673.0,20712.0,30645.0,56.0
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188353.0,150188.0,180600.0,47.0
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199528.0,1,160160.0,128110.0,154077.0,23.0


In [300]:
grouped_df.dtypes

기획년도                    int64
주차             datetime64[ns]
카테고리                   object
라인                     object
시즌이월                   object
시즌                     object
복종                     object
소품종                    object
성별                     object
총입고수량                   int64
판매수량                    int64
판매액                     int64
평균택가                    int64
평균원가                    int64
총입고원가                   int64
총입고택가                   int64
매출원가계                   int64
판매택가계                   int64
주차별_평균_실판매가           float64
월                       int32
월별_평균_실판매가            float64
시즌별_평균_실판매가           float64
실판매가                  float64
할인율                   float64
dtype: object

### 중간확인

In [301]:
grouped_df[grouped_df['할인율'] < 0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율
23,2021,2021-01-03,사계절_소품_벨트_ZB,ZB,01_시즌,사계절,소품,벨트,1:남성,8650,184,7443910,39900,7684,66462275,345135000,1413764,7341600,38215.0,1,37842.0,38165.0,40456.0,-1.0
55,2021,2021-01-10,봄_코트_싱글코트_ZB,ZB,01_시즌,봄,코트,싱글코트,1:남성,10358,-21,-7550500,249000,25439,263497162,2579142000,-534219,-5229000,359548.0,1,178906.0,119035.0,359548.0,-44.0
91,2021,2021-01-17,봄_코트_더블코트_ZB,ZB,01_시즌,봄,코트,더블코트,1:남성,3377,-12,-5411000,329000,34815,117570255,1111033000,-417780,-3948000,450917.0,1,243505.0,144883.0,450917.0,-37.0
97,2021,2021-01-17,사계절_소품_벨트_ZB,ZB,01_시즌,사계절,소품,벨트,1:남성,8650,155,6325950,39900,7684,66462275,345135000,1190942,6184500,38656.0,1,37842.0,38165.0,40813.0,-2.0
137,2021,2021-01-24,사계절_소품_벨트_ZB,ZB,01_시즌,사계절,소품,벨트,1:남성,8650,166,6951880,39900,7684,66462275,345135000,1275461,6623400,38624.0,1,37842.0,38165.0,41879.0,-5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20349,2024,2024-11-17,사계절_소품_양말_ZB,ZB,01_시즌,사계절,소품,양말,1:남성,16565,92,565240,5700,1192,19745480,94420500,109664,524400,6183.0,11,5817.0,5622.0,6144.0,-8.0
20973,2024,2024-12-15,사계절_소품_양말_ZB,ZB,01_시즌,사계절,소품,양말,1:남성,16565,85,487500,5700,1192,19745480,94420500,101320,484500,5700.0,12,5666.0,5622.0,5735.0,-1.0
21129,2024,2024-12-22,사계절_소품_양말_ZB,ZB,01_시즌,사계절,소품,양말,1:남성,16565,100,581590,5700,1192,19745480,94420500,119200,570000,5741.0,12,5666.0,5622.0,5816.0,-2.0
21197,2024,2024-12-29,가을_우븐 셔츠_캐쥬얼셔츠_ZB,ZB,02_이월,가을,우븐 셔츠,캐쥬얼셔츠,1:남성,9036,-1,-2179253,79000,8819,79688484,713844000,-8819,-79000,42442.0,12,24786.0,24786.0,2179253.0,-2659.0


## 누적 컬럼 생성

In [302]:
grouped_df['누적판매수량'] = grouped_df.groupby(['기획년도', '카테고리'])['판매수량'].cumsum()
grouped_df['누적판매액'] = grouped_df.groupby(['기획년도','카테고리'])['판매액'].cumsum()
grouped_df['누적매출원가'] = grouped_df.groupby(['기획년도','카테고리'])['매출원가계'].cumsum()
grouped_df['누적판매택가'] = grouped_df.groupby(['기획년도','카테고리'])['판매택가계'].cumsum()
        
grouped_df['누적판매율'] = (grouped_df['누적판매수량'] / grouped_df['총입고수량']* 100).round(2)
grouped_df['ROI'] = ((grouped_df['누적판매액']/1.1 - grouped_df['누적매출원가'])/grouped_df['총입고원가']).round(2)
grouped_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693.0,986743.0,1.0,53,52297400,10838500,52947000,3.31,0.11
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41774.0,27610.0,47335.0,32.0,401,18981424,3560078,28029900,2.00,0.08
2,2021,2021-01-03,봄_니트 셔츠_라운드_ZE,ZE,01_시즌,봄,니트 셔츠,라운드,1:남성,17148,7,214516,69900,6179,105957492,1198645200,43253,489300,30645.0,1,24673.0,20712.0,30645.0,56.0,7,214516,43253,489300,0.04,0.00
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188353.0,150188.0,180600.0,47.0,109,19685400,6120386,36951000,2.49,0.05
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199528.0,1,160160.0,128110.0,154077.0,23.0,194,29890941,7873141,38606000,2.32,0.06


In [ ]:
import pandas as pd

# 데이터 불러오기
file_path = "./row_data/서울_기온강수량적설량(21~24년).csv"
weather_df = pd.read_csv(file_path, encoding="cp949", parse_dates=["일시"])

# 필요없는 컬럼 제거
weather_df = weather_df[['일시', '평균기온(°C)', '1시간 최다강수량(mm)', '일 최심신적설(cm)']]

# '평균 날씨' 컬럼 생성 함수
def classify_weather(row):
    if pd.isna(row['1시간 최다강수량(mm)']) and pd.isna(row['일 최심신적설(cm)']):
        return '맑음'
    if pd.notna(row['1시간 최다강수량(mm)']) and row['1시간 최다강수량(mm)'] <= 1:
        return '흐림'
    if pd.isna(row['일 최심신적설(cm)']):
        if row['1시간 최다강수량(mm)'] > 15:
            return '강한 비'
        elif row['1시간 최다강수량(mm)'] > 0:
            return '비'
    if row['일 최심신적설(cm)'] > 5:
        return '강한 눈'
    if row['평균기온(°C)'] < 5:
        return '눈'
    return '진눈깨비'

# 일별 '평균 날씨' 컬럼 생성
weather_df['평균 날씨'] = weather_df.apply(classify_weather, axis=1)

# 주간 평균 기온 계산 (2021-01-03부터 시작)
weather_df.set_index('일시', inplace=True)
weekly_temp_df = weather_df[['평균기온(°C)']].resample('W', label='left', closed='left').mean().round(1)

# 주차별 날씨 상태 개수 카운트 (groupby() 활용하여 MultiIndex 생성 후 unstack)
weekly_weather_counts = (
    weather_df.groupby([pd.Grouper(freq='W', label='left', closed='left'), '평균 날씨'])
    .size()
    .unstack(fill_value=0)
)

# '강한 비' + '강한 눈'을 합쳐 '악천후일수' 생성
weekly_weather_counts['악천후일수'] = weekly_weather_counts[['강한 비', '강한 눈']].sum(axis=1)

# 인덱스 정리 (2021-01-03부터 시작하도록 조정)
weekly_weather_df = weekly_weather_counts.reset_index()
weekly_weather_df.rename(columns={'index': '주차'}, inplace=True)
weekly_weather_df['주차'] = pd.date_range(start="2021-01-03", periods=len(weekly_weather_df), freq='W')

# 주차별 평균 기온 추가
weekly_weather_df = weekly_weather_df.merge(weekly_temp_df, left_on='주차', right_index=True, how='left')

# 컬럼명 변경
weekly_weather_df = weekly_weather_df.reset_index()
weekly_weather_df.rename(columns={'평균기온(°C)': '평균기온(도)'}, inplace=True)

# 컬럼 순서 변경
columns_order = ['주차', '맑음', '흐림', '비', '강한 비', '눈', '강한 눈', '진눈깨비', '악천후일수', '평균기온(도)']
weekly_weather_df = weekly_weather_df[columns_order]

# grouped_df와 병합
grouped_df = grouped_df.merge(weekly_weather_df, on='주차', how='left')
grouped_df

## EDA용 csv 생성

In [303]:
# grouped_df.to_csv("EDA_df_0314.csv", index=False, encoding="utf-8-sig")

## 모델링용 전처리(특정 라인(ZD, ZE, ZF) 제거, 시즌이월 '이월' 제거,  '소품', '언더웨어' 제거)

In [304]:
model_df = grouped_df[grouped_df['시즌이월'] != '02_이월']
model_df = model_df[~model_df['카테고리'].str.contains('ZD|ZE|ZF|소품|언더웨어', na=False)]
model_df.head()

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693.0,986743.0,1.0,53,52297400,10838500,52947000,3.31,0.11
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41774.0,27610.0,47335.0,32.0,401,18981424,3560078,28029900,2.00,0.08
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188353.0,150188.0,180600.0,47.0,109,19685400,6120386,36951000,2.49,0.05
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199528.0,1,160160.0,128110.0,154077.0,23.0,194,29890941,7873141,38606000,2.32,0.06
5,2021,2021-01-03,봄_수트_블레이져(수트)_ZC,ZC,01_시즌,봄,수트,블레이져(수트),1:남성,2868,33,5652700,239000,46064,132112508,685452000,1520123,7887000,175804.0,1,160403.0,133058.0,171294.0,28.0,33,5652700,1520123,7887000,1.15,0.03


## 수량 이상치 확인

In [305]:
# 판매수량이 음수인 값만 필터링
negative_sales_df = model_df[model_df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 1.5 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")

이상치 기점: -246.0, 3 IQR 이상인 이상치 개수: 20


In [306]:
model_df[model_df['실판매가']<=0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI


In [307]:
model_df[model_df['실판매가'].isna()]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI


In [308]:
model_df[model_df['할인율']<=0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI
20,2021,2021-01-03,봄_코트_더블코트_ZB,ZB,01_시즌,봄,코트,더블코트,1:남성,3340,45,14805000,329000,34814,116278760,1098860000,1566630,14805000,329000.0,1,243505.0,144883.0,329000.0,0.0,45,14805000,1566630,14805000,1.35,0.10
22,2021,2021-01-03,봄_코트_싱글코트_ZB,ZB,01_시즌,봄,코트,싱글코트,1:남성,3503,79,19646100,249000,25314,88674942,872247000,1999806,19671000,248685.0,1,178906.0,119035.0,248685.0,0.0,79,19646100,1999806,19671000,2.26,0.18
55,2021,2021-01-10,봄_코트_싱글코트_ZB,ZB,01_시즌,봄,코트,싱글코트,1:남성,10358,-21,-7550500,249000,25439,263497162,2579142000,-534219,-5229000,359548.0,1,178906.0,119035.0,359548.0,-44.0,58,12095600,1465587,14442000,0.56,0.04
91,2021,2021-01-17,봄_코트_더블코트_ZB,ZB,01_시즌,봄,코트,더블코트,1:남성,3377,-12,-5411000,329000,34815,117570255,1111033000,-417780,-3948000,450917.0,1,243505.0,144883.0,450917.0,-37.0,78,16465000,2715480,25662000,2.31,0.10
105,2021,2021-01-17,사계절_자켓_싱글재킷_ZB,ZB,01_시즌,사계절,자켓,싱글재킷,1:남성,5145,0,0,199000,23618,121514610,1023855000,0,0,NaN,1,199000.0,110719.0,199000.0,0.0,0,0,0,0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18781,2024,2024-09-08,겨울_점퍼_패딩점퍼_ZB,ZB,01_시즌,겨울,점퍼,패딩점퍼,1:남성,8030,-2,-448200,224000,32296,259336880,1798720000,-64592,-448000,224100.0,9,180525.0,139210.0,224100.0,-0.0,4,747000,143560,1046000,0.05,0.00
19156,2024,2024-09-22,여름_점퍼_점퍼_ZB,ZB,01_시즌,여름,점퍼,점퍼,1:남성,11515,2,528000,249000,24849,286136235,2867235000,49698,498000,100209.0,9,101424.0,113074.0,264000.0,-6.0,8889,952115530,220882705,2213361000,77.19,2.25
19195,2024,2024-09-29,겨울_사파리_패딩사파리_ZB,ZB,01_시즌,겨울,사파리,패딩사파리,1:남성,5847,-3,-1197000,399000,25274,147774154,2332953000,-75820,-1197000,399000.0,9,399000.0,201703.0,399000.0,0.0,-3,-1197000,-75820,-1197000,-0.05,-0.01
19359,2024,2024-10-06,겨울_스웨터_Turtle_ZB,ZB,01_시즌,겨울,스웨터,Turtle,1:남성,21528,80,9714900,119000,15076,324563304,2561832000,1206107,9520000,119937.0,10,97449.0,68760.0,121436.0,-2.0,121,14637500,1874325,16039000,0.56,0.04


In [309]:
model_df[model_df['할인율'].isna()]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI


In [310]:
model_df.dtypes

기획년도                    int64
주차             datetime64[ns]
카테고리                   object
라인                     object
시즌이월                   object
시즌                     object
복종                     object
소품종                    object
성별                     object
총입고수량                   int64
판매수량                    int64
판매액                     int64
평균택가                    int64
평균원가                    int64
총입고원가                   int64
총입고택가                   int64
매출원가계                   int64
판매택가계                   int64
주차별_평균_실판매가           float64
월                       int32
월별_평균_실판매가            float64
시즌별_평균_실판매가           float64
실판매가                  float64
할인율                   float64
누적판매수량                  int64
누적판매액                   int64
누적매출원가                  int64
누적판매택가                  int64
누적판매율                 float64
ROI                   float64
dtype: object

In [311]:
def update_data(df, input_rows, output_rows):
    df = df.copy()  # 원본 데이터 보호

    # Step 1: output 행 값 변경 (input과 output을 합쳐서 덮어쓰기)
    for i in range(len(input_rows)):
        input_date, category = input_rows[i]
        output_date, _ = output_rows[i]

        # input과 output 행 찾기
        mask_input = (df['주차'] == input_date) & (df['카테고리'] == category)
        mask_output = (df['주차'] == output_date) & (df['카테고리'] == category)

        # input 행과 output 행 판매수량 합치기
        df.loc[mask_output, '판매수량'] = (
            df.loc[mask_input, '판매수량'].values +
            df.loc[mask_output, '판매수량'].values
        )
        
        # 판매액, 매출원가, 판매택가 재계산
        df.loc[mask_output, '판매액'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '주차별_평균_실판매가']).round(0)
        df.loc[mask_output, '매출원가계'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '평균원가']).round(0)
        df.loc[mask_output, '판매택가계'] = (df.loc[mask_output, '판매수량'] * df.loc[mask_output, '평균택가']).round(0)
        
        # 제품실판가, 할인율 재계산
        df.loc[mask_output, '실판매가'] = df.loc[mask_output, '판매액'] / df.loc[mask_output, '판매수량'] #null 값 있음
        df.loc[mask_output, '할인율'] = (df.loc[mask_output, '평균택가'] - df.loc[mask_output, '실판매가']) * 100 / df.loc[mask_output, '평균택가'] #null 값 있음
        
    # Step 2: input 행 값 변경 (이전, 이후 주차 평균으로 대체) -> 0으로 변경
    for input_date, category in input_rows:
        mask_input = (df['주차'] == input_date) & (df['카테고리'] == category)

        # input 행 값 변경
        df.loc[mask_input, ['판매액', '판매수량', '매출원가계', '판매택가계']] = 0

        # 제품실판가,할인율
        # 제품실판가, 할인율을 NaN으로 설정
        df.loc[mask_input, ['실판매가', '할인율']] = np.nan
        
    # 누적 집계값 업데이트
    df['누적판매수량'] = df.groupby(['기획년도', '카테고리'])['판매수량'].cumsum()
    df['누적판매액'] = df.groupby(['기획년도','카테고리'])['판매액'].cumsum()
    df['누적매출원가'] = df.groupby(['기획년도','카테고리'])['매출원가계'].cumsum()
    df['누적판매택가'] = df.groupby(['기획년도','카테고리'])['판매택가계'].cumsum()
        
    # 판매율 / ROI 업데이트
    df['누적판매율'] = (df['누적판매수량'] / df['총입고수량']* 100).round(2)
    df['ROI'] = ((df['누적판매액']/1.1 - df['누적매출원가'])/df['총입고원가']).round(2)
        
    return df

# 실행
input_rows = [
    ('2022-09-18', '겨울_스웨터_라운드_ZB'),
    ('2024-04-07', '봄_자켓_싱글재킷_ZB'),
    ('2022-07-31', '여름_니트 셔츠_라운드_ZB'),
    ('2022-09-25', '여름_니트 셔츠_라운드_ZB'),
    ('2022-07-24', '여름_자켓_싱글재킷_ZB'),
    ('2022-08-07', '여름_자켓_싱글재킷_ZB'),
    ('2022-12-11', '사계절_우븐 셔츠_드레스셔츠_ZB'),
    ('2023-10-08', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-15', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-29', '사계절_데님_데님팬츠_ZB'),
    ('2023-12-10', '사계절_데님_데님팬츠_ZB'),
    ('2024-12-29', '사계절_데님_데님팬츠_ZB'),
    ('2023-07-09', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-16', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-23', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-08-20', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-09-24', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-12-10', '사계절_니트 셔츠_라운드_ZB')
]

output_rows = [
    ('2022-09-11', '겨울_스웨터_라운드_ZB'),
    ('2024-03-31', '봄_자켓_싱글재킷_ZB'),
    ('2022-07-10', '여름_니트 셔츠_라운드_ZB'),
    ('2022-09-18', '여름_니트 셔츠_라운드_ZB'),
    ('2022-07-10', '여름_자켓_싱글재킷_ZB'),
    ('2022-07-31', '여름_자켓_싱글재킷_ZB'),
    ('2022-12-04', '사계절_우븐 셔츠_드레스셔츠_ZB'),
    ('2023-10-01', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-01', '사계절_데님_데님팬츠_ZB'),
    ('2023-10-22', '사계절_데님_데님팬츠_ZB'),
    ('2023-12-03', '사계절_데님_데님팬츠_ZB'),
    ('2024-12-22', '사계절_데님_데님팬츠_ZB'),
    ('2023-06-18', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-07-02', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-06-18', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-08-13', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-09-03', '사계절_니트 셔츠_라운드_ZB'),
    ('2023-12-03', '사계절_니트 셔츠_라운드_ZB')
]

df_updated = update_data(model_df, input_rows, output_rows)

In [312]:
df_updated

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693.0,986743.0,1.0,53,52297400,10838500,52947000,3.31,0.11
1,2021,2021-01-03,봄_니트 셔츠_라운드_ZB,ZB,01_시즌,봄,니트 셔츠,라운드,1:남성,20076,401,18981424,69900,8878,178234728,1403312400,3560078,28029900,47335.0,1,41774.0,27610.0,47335.0,32.0,401,18981424,3560078,28029900,2.00,0.08
3,2021,2021-01-03,봄_수트_블레이져(수트)_ZA,ZA,01_시즌,봄,수트,블레이져(수트),1:남성,4378,109,19685400,339000,56150,245826159,1484142000,6120386,36951000,207972.0,1,188353.0,150188.0,180600.0,47.0,109,19685400,6120386,36951000,2.49,0.05
4,2021,2021-01-03,봄_수트_블레이져(수트)_ZB,ZB,01_시즌,봄,수트,블레이져(수트),1:남성,8372,194,29890941,199000,40583,339762550,1666028000,7873141,38606000,199528.0,1,160160.0,128110.0,154077.0,23.0,194,29890941,7873141,38606000,2.32,0.06
5,2021,2021-01-03,봄_수트_블레이져(수트)_ZC,ZC,01_시즌,봄,수트,블레이져(수트),1:남성,2868,33,5652700,239000,46064,132112508,685452000,1520123,7887000,175804.0,1,160403.0,133058.0,171294.0,28.0,33,5652700,1520123,7887000,1.15,0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21287,2024,2024-12-29,사계절_수트_블레이져(수트)_ZB,ZB,01_시즌,사계절,수트,블레이져(수트),1:남성,21880,145,17004054,259000,42625,932635000,5666920000,6180625,37555000,98988.0,12,105789.0,131261.0,117269.0,55.0,18670,2617630459,797816000,4835530000,85.33,1.70
21288,2024,2024-12-29,사계절_수트_블레이져(수트)_ZC,ZC,01_시즌,사계절,수트,블레이져(수트),1:남성,7019,39,5152900,259000,44254,310618826,1817921000,1725906,10101000,138896.0,12,140293.0,148324.0,132126.0,49.0,5833,863955390,258801883,1510747000,83.10,1.70
21289,2024,2024-12-29,사계절_수트_수트팬츠_ZB,ZB,01_시즌,사계절,수트,수트팬츠,1:남성,26254,170,11050814,139000,20924,549338696,3649306000,3557080,23630000,56260.0,12,56913.0,69377.0,65005.0,53.0,20654,1519812174,429547734,2870906000,78.67,1.73
21290,2024,2024-12-29,사계절_수트_수트팬츠_ZC,ZC,01_시즌,사계절,수트,수트팬츠,1:남성,8424,51,3838230,139000,21438,180589500,1170936000,1093312,7089000,78054.0,12,77179.0,78577.0,75259.0,46.0,5966,466517905,128160254,829274000,70.82,1.64


In [313]:
df_updated.to_csv("model_df_0314.csv", index=False, encoding="utf-8-sig")